# Кластерний Аналіз - UMAP + HDBSCAN

Виявлення природних груп у відгуках для пошуку підозрілих патернів

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import hdbscan
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

## 1. Завантаження даних

In [ ]:
# Завантажуємо датасет
df = pd.read_csv('../../data/doctors_reviews_engineered.csv')
print(f"Завантажено {len(df):,} відгуків")

# Завантажуємо ембедінги
embeddings = np.load('../../data/xlm_roberta_embeddings.npy')
text_indices = np.load('../../data/text_indices.npy')
print(f"Завантажено ембедінги: {embeddings.shape}")
print(f"Індекси: {len(text_indices):,}")

# Створюємо датасет тільки з текстовими відгуками
df_with_embeddings = df.loc[text_indices].copy()
print(f"Датасет для кластеризації: {len(df_with_embeddings):,} записів")

## 2. UMAP - Dimension Reduction

In [ ]:
# UMAP для зниження розмірності до 2D (для візуалізації)
print("Застосовуємо UMAP для 2D візуалізації...")
reducer_2d = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='cosine',
    random_state=42,
    verbose=True
)
embeddings_2d = reducer_2d.fit_transform(embeddings)
print(f"2D ембедінги: {embeddings_2d.shape}")

In [ ]:
# UMAP для зниження розмірності до 5D (для кластеризації - краща якість)
print("Застосовуємо UMAP для 5D кластеризації...")
reducer_5d = umap.UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=5,
    metric='cosine',
    random_state=42,
    verbose=True
)
embeddings_5d = reducer_5d.fit_transform(embeddings)
print(f"5D ембедінги: {embeddings_5d.shape}")

In [ ]:
# Додаємо 2D координати до датасету
df_with_embeddings['umap_x'] = embeddings_2d[:, 0]
df_with_embeddings['umap_y'] = embeddings_2d[:, 1]

print("UMAP координати додано до датасету")

## 3. Базова візуалізація UMAP

In [ ]:
# Візуалізація всіх точок
plt.figure(figsize=(14, 10))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.3, s=5, c='steelblue')
plt.title('UMAP Projection всіх відгуків', fontsize=16)
plt.xlabel('UMAP Dimension 1', fontsize=12)
plt.ylabel('UMAP Dimension 2', fontsize=12)
plt.tight_layout()
plt.savefig('../../data/umap_all_reviews.png', dpi=150)
plt.show()

## 4. HDBSCAN Кластеризація

In [ ]:
# HDBSCAN на 5D ембедінгах для кращої якості
print("Застосовуємо HDBSCAN кластеризацію...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,    # Мінімальний розмір кластера
    min_samples=10,     # Мінімальна кількість сусідів
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(embeddings_5d)

# Додаємо мітки кластерів до датасету
df_with_embeddings['cluster'] = cluster_labels

# Outlier scores (чим вище - тим більш аномальний)
df_with_embeddings['outlier_score'] = clusterer.outlier_scores_

print(f"\nКластеризація завершена!")
print(f"Унікальних кластерів: {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}")
print(f"Шуму (outliers, кластер -1): {(cluster_labels == -1).sum():,} ({(cluster_labels == -1).sum()/len(cluster_labels)*100:.2f}%)")

In [ ]:
# Статистика по кластерах
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("\nРозподіл по кластерах:")
print(cluster_counts.head(20))

# Візуалізація розподілу
plt.figure(figsize=(12, 6))
cluster_counts[cluster_counts.index != -1].plot(kind='bar', color='steelblue')
plt.xlabel('Кластер')
plt.ylabel('Кількість відгуків')
plt.title('Розподіл відгуків по кластерах')
plt.tight_layout()
plt.savefig('../../data/cluster_distribution.png', dpi=150)
plt.show()

## 5. Візуалізація кластерів на UMAP

In [ ]:
# Візуалізація кластерів
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Всі кластери
scatter = axes[0].scatter(
    embeddings_2d[:, 0], 
    embeddings_2d[:, 1], 
    c=cluster_labels, 
    cmap='tab20', 
    alpha=0.5, 
    s=10
)
axes[0].set_title('UMAP з кластерами (всі)', fontsize=14)
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
plt.colorbar(scatter, ax=axes[0], label='Кластер')

# Тільки outliers
is_outlier = cluster_labels == -1
axes[1].scatter(
    embeddings_2d[~is_outlier, 0], 
    embeddings_2d[~is_outlier, 1], 
    c='lightgray', 
    alpha=0.3, 
    s=5,
    label='Нормальні кластери'
)
axes[1].scatter(
    embeddings_2d[is_outlier, 0], 
    embeddings_2d[is_outlier, 1], 
    c='red', 
    alpha=0.6, 
    s=20,
    label='Outliers (підозрілі)'
)
axes[1].set_title('UMAP - Outliers vs Нормальні', fontsize=14)
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../data/umap_clusters_visualization.png', dpi=150)
plt.show()

## 6. Аналіз характеристик кластерів

In [ ]:
# Порівнюємо характеристики різних кластерів
cluster_stats = df_with_embeddings.groupby('cluster').agg({
    'text_length': 'mean',
    'word_count': 'mean',
    'is_empty': 'sum',
    'is_anonymous': 'mean',
    'doctor_total_reviews': 'mean',
    'Коментар': 'count'
}).round(2)

cluster_stats.columns = ['Сер. довжина', 'Сер. слів', 'Порожніх', 'Частка анонімних', 'Відгуків на лікаря', 'Кількість']
print("\nХарактеристики топ кластерів:")
print(cluster_stats.sort_values('Кількість', ascending=False).head(10))

## Аналіз найпідозріліших відгуків

In [ ]:
# Топ outliers за outlier score
top_outliers = df_with_embeddings.nlargest(100, 'outlier_score')[[
    'Коментар', 'Ім\'я лікаря', 'Ім\'я коментатора', 'outlier_score', 
    'text_length', 'word_count', 'is_anonymous', 'cluster'
]]

print("\nТоп-20 найпідозріліших відгуків за outlier score:")
print(top_outliers.head(20))

In [ ]:
# Розподіл outlier scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_with_embeddings['outlier_score'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Outlier Score')
axes[0].set_ylabel('Частота')
axes[0].set_title('Розподіл Outlier Scores')
axes[0].axvline(df_with_embeddings['outlier_score'].quantile(0.95), 
                color='red', linestyle='--', label='95-й перцентиль')
axes[0].legend()

# Boxplot
axes[1].boxplot(df_with_embeddings['outlier_score'], vert=False)
axes[1].set_xlabel('Outlier Score')
axes[1].set_title('Boxplot Outlier Scores')

plt.tight_layout()
plt.savefig('../../data/outlier_scores_distribution.png', dpi=150)
plt.show()

Аналіз дрібних кластерів (можливо підозрілі шаблони)

In [ ]:
# Кластери з малою кількістю відгуків можуть бути підозрілими шаблонами
small_clusters = cluster_counts[(cluster_counts < 200) & (cluster_counts.index != -1)]
print(f"\nМалих кластерів (< 200 відгуків): {len(small_clusters)}")

# Подивимось на приклади з малих кластерів
for cluster_id in small_clusters.index[:5]:
    cluster_reviews = df_with_embeddings[df_with_embeddings['cluster'] == cluster_id]
    print(f"\n{'='*80}")
    print(f"Кластер {cluster_id} - {len(cluster_reviews)} відгуків")
    print(f"Середня довжина: {cluster_reviews['text_length'].mean():.0f} символів")
    print(f"Анонімних: {cluster_reviews['is_anonymous'].mean()*100:.1f}%")
    print("\nПриклади відгуків:")
    for i, review in enumerate(cluster_reviews['Коментар'], 1):
        print(f"{i}. {review}")

## 9. Візуалізація за довжиною тексту

In [ ]:
# Колоруємо за довжиною тексту
plt.figure(figsize=(14, 10))
scatter = plt.scatter(
    embeddings_2d[:, 0], 
    embeddings_2d[:, 1], 
    c=df_with_embeddings['text_length'],
    cmap='viridis',
    alpha=0.5,
    s=10
)
plt.colorbar(scatter, label='Довжина тексту (символів)')
plt.title('UMAP - колорування за довжиною тексту', fontsize=16)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.savefig('../../data/umap_by_text_length.png', dpi=150)
plt.show()

## 10. Збереження результатів

In [ ]:
# Зберігаємо датасет з кластерами та outlier scores
df_with_embeddings.to_csv('../../data/reviews_with_clusters.csv', index=False)
print(f"Датасет з кластерами збережено: {len(df_with_embeddings):,} записів")

# Зберігаємо UMAP координати окремо
np.save('../../data/umap_2d.npy', embeddings_2d)
np.save('../../data/umap_5d.npy', embeddings_5d)
print("UMAP координати збережено")